# 05 - X-Ray Dataset Exploration

This notebook explores the COVID-19 Radiography Dataset for chest X-ray classification.

**Goals:**
1. Download and load the dataset
2. Understand the data structure
3. Visualize sample images
4. Analyze class distribution
5. Prepare for model training

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter
import yaml

print(f"Project root: {project_root}")

## 1. Dataset Information

We'll use the **COVID-19 Radiography Database** from Kaggle.

**Dataset Details:**
- ~21,000 chest X-ray images
- 4 classes: COVID, Normal, Viral Pneumonia, Lung Opacity
- High-quality images (299x299 or higher)

**Download Instructions:**
1. Go to: https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database
2. Download and extract to `data/xray/`
3. Or use Kaggle API: `kaggle datasets download -d tawsifurrahman/covid19-radiography-database`

In [ ]:
# Load configuration
config_path = project_root / "config_xray.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"  Model: {config['model']['backbone']}")
print(f"  Classes: {config['dataset']['classes']}")
print(f"  Image size: {config['preprocessing']['image_size']}")

In [ ]:
# Check if dataset exists
data_dir = project_root / "data" / "xray"
data_dir.mkdir(parents=True, exist_ok=True)

# Expected structure after download
expected_path = data_dir / "COVID-19_Radiography_Dataset"

if expected_path.exists():
    print(f"Dataset found at: {expected_path}")
    dataset_path = expected_path
else:
    print(f"Dataset NOT found at: {expected_path}")
    print("\n" + "="*60)
    print("DOWNLOAD INSTRUCTIONS")
    print("="*60)
    print("\nOption 1: Manual Download")
    print("1. Go to: https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database")
    print("2. Click 'Download' (requires Kaggle account)")
    print(f"3. Extract contents to: {data_dir}/")
    print("\nOption 2: Kaggle API")
    print("pip install kaggle")
    print("kaggle datasets download -d tawsifurrahman/covid19-radiography-database")
    print(f"unzip covid19-radiography-database.zip -d {data_dir}/")
    print("\n" + "="*60)
    dataset_path = None

## 2. Explore Dataset Structure

In [ ]:
if dataset_path and dataset_path.exists():
    print("Dataset Structure:")
    print("-" * 40)
    
    class_counts = {}
    for class_dir in sorted(dataset_path.iterdir()):
        if class_dir.is_dir():
            # Count images in each class
            images_dir = class_dir / "images"
            if images_dir.exists():
                count = len(list(images_dir.glob("*.png"))) + len(list(images_dir.glob("*.jpg")))
            else:
                count = len(list(class_dir.glob("*.png"))) + len(list(class_dir.glob("*.jpg")))
            class_counts[class_dir.name] = count
            print(f"  {class_dir.name}: {count:,} images")
    
    total = sum(class_counts.values())
    print(f"\nTotal images: {total:,}")
else:
    print("Dataset not found. Please download first.")
    # Create mock data for demonstration
    class_counts = {'COVID': 3616, 'Normal': 10192, 'Viral Pneumonia': 1345}
    print("\nExpected dataset structure:")
    for cls, count in class_counts.items():
        print(f"  {cls}: ~{count:,} images")

## 3. Visualize Class Distribution

In [ ]:
# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

classes = list(class_counts.keys())
counts = list(class_counts.values())
colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']

# Bar chart
axes[0].bar(classes, counts, color=colors[:len(classes)])
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Images')
axes[0].set_title('X-Ray Class Distribution')
axes[0].tick_params(axis='x', rotation=15)

# Add count labels
for i, (cls, count) in enumerate(zip(classes, counts)):
    axes[0].text(i, count + 100, f'{count:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(counts, labels=classes, autopct='%1.1f%%', colors=colors[:len(classes)])
axes[1].set_title('Class Percentage Distribution')

plt.tight_layout()
plt.show()

## 4. Visualize Sample Images

In [ ]:
def get_sample_images(dataset_path, class_name, n=3):
    """Get n sample images from a class."""
    class_dir = dataset_path / class_name
    images_dir = class_dir / "images" if (class_dir / "images").exists() else class_dir
    
    image_files = list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg"))
    samples = image_files[:n] if len(image_files) >= n else image_files
    
    images = []
    for img_path in samples:
        img = Image.open(img_path).convert('RGB')
        images.append(img)
    
    return images

In [ ]:
if dataset_path and dataset_path.exists():
    # Get classes that exist
    available_classes = [d.name for d in dataset_path.iterdir() if d.is_dir()]
    display_classes = available_classes[:3]  # Show first 3 classes
    
    fig, axes = plt.subplots(len(display_classes), 3, figsize=(12, 4*len(display_classes)))
    
    for i, class_name in enumerate(display_classes):
        images = get_sample_images(dataset_path, class_name, n=3)
        for j, img in enumerate(images):
            if len(display_classes) == 1:
                ax = axes[j]
            else:
                ax = axes[i, j]
            ax.imshow(img, cmap='gray')
            ax.set_title(f"{class_name}")
            ax.axis('off')
    
    plt.suptitle('Sample X-Ray Images by Class', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Dataset not available for visualization.")
    print("\nExpected sample images:")
    print("- COVID: Chest X-rays showing ground-glass opacities")
    print("- Normal: Clear lung fields")
    print("- Viral Pneumonia: Diffuse bilateral infiltrates")

## 5. Image Statistics

In [ ]:
if dataset_path and dataset_path.exists():
    # Analyze image properties
    print("Analyzing image properties...")
    
    sizes = []
    for class_dir in dataset_path.iterdir():
        if class_dir.is_dir():
            images_dir = class_dir / "images" if (class_dir / "images").exists() else class_dir
            image_files = list(images_dir.glob("*.png"))[:50]  # Sample 50 per class
            
            for img_path in image_files:
                img = Image.open(img_path)
                sizes.append(img.size)
    
    if sizes:
        widths, heights = zip(*sizes)
        print(f"\nImage Size Statistics (sampled {len(sizes)} images):")
        print(f"  Width - Min: {min(widths)}, Max: {max(widths)}, Mean: {np.mean(widths):.0f}")
        print(f"  Height - Min: {min(heights)}, Max: {max(heights)}, Mean: {np.mean(heights):.0f}")
        print(f"  Most common size: {Counter(sizes).most_common(1)[0][0]}")
else:
    print("Dataset not available for analysis.")
    print("\nTypical COVID-19 Radiography Dataset properties:")
    print("  - Image size: 299x299 pixels")
    print("  - Format: PNG")
    print("  - Color: Grayscale (converted to RGB for models)")

## 6. Prepare Data Splits

In [ ]:
# Calculate recommended splits
total_images = sum(class_counts.values())
train_ratio = config['dataset']['train_ratio']
val_ratio = config['dataset']['val_ratio']
test_ratio = config['dataset']['test_ratio']

print("\n" + "="*50)
print("RECOMMENDED DATA SPLITS")
print("="*50)
print(f"\nTotal images: {total_images:,}")
print(f"\nSplit configuration:")
print(f"  Train: {train_ratio*100:.0f}% = ~{int(total_images*train_ratio):,} images")
print(f"  Validation: {val_ratio*100:.0f}% = ~{int(total_images*val_ratio):,} images")
print(f"  Test: {test_ratio*100:.0f}% = ~{int(total_images*test_ratio):,} images")

print("\nPer-class split (approximate):")
for cls, count in class_counts.items():
    print(f"  {cls}:")
    print(f"    Train: {int(count*train_ratio):,}")
    print(f"    Val: {int(count*val_ratio):,}")
    print(f"    Test: {int(count*test_ratio):,}")

## 7. Data Imbalance Considerations

In [ ]:
# Analyze class imbalance
print("\n" + "="*50)
print("CLASS IMBALANCE ANALYSIS")
print("="*50)

total = sum(class_counts.values())
max_count = max(class_counts.values())
min_count = min(class_counts.values())

print(f"\nImbalance ratio: {max_count/min_count:.2f}:1")
print(f"Largest class: {max(class_counts, key=class_counts.get)} ({max_count:,})")
print(f"Smallest class: {min(class_counts, key=class_counts.get)} ({min_count:,})")

print("\nRecommended strategies:")
print("  1. Weighted loss function (class weights inversely proportional to frequency)")
print("  2. Oversampling minority classes")
print("  3. Data augmentation on minority classes")
print("  4. Focal loss for hard examples")

# Calculate class weights
print("\nSuggested class weights:")
for cls, count in class_counts.items():
    weight = total / (len(class_counts) * count)
    print(f"  {cls}: {weight:.3f}")

## 8. Summary and Next Steps

In [ ]:
print("\n" + "#"*60)
print("# DATASET EXPLORATION SUMMARY")
print("#"*60)

print(f"\nDataset: COVID-19 Radiography Database")
print(f"Total images: {total_images:,}")
print(f"Classes: {len(class_counts)}")

print(f"\nClass distribution:")
for cls, count in class_counts.items():
    pct = count/total_images*100
    print(f"  {cls}: {count:,} ({pct:.1f}%)")

print(f"\nNext Steps:")
print(f"  1. Run 06_xray_training.ipynb to train the model")
print(f"  2. Or use: python src/image_train.py --config config_xray.yaml")
print(f"  3. Model will be saved to: outputs/xray_models/")

print("\n" + "#"*60)